# 02 从 loss 到「参数真的变了」

上一课（01）走完了：**原文 → token id → 对齐 → logits → softmax → loss**。

今天只往前走一步，回答日志里记的那个问题：

> `loss.backward()` 到底产生了什么信号？

**今天 30 分钟，只有两格。**

| 格 | 一句话 | 预计 |
| --- | --- | --- |
| 格 1 | `backward()` 把什么写进了 `.grad` | 12 分钟 |
| 格 2 | `.grad` 的正负号是什么意思，怎么用它改参数 | 15 分钟 |

**规则：每一格先看「先预测」，把答案写在纸上或用脑子想定，再运行代码。**

预测错了比预测对了更有价值——错的预测正好指出哪个直觉是歪的。

---

**知识点标注**

- 行业状态：**基础标准**。反向传播和梯度下降是 1986 年就在用的东西，今天所有 PyTorch 训练都跑在这套机制上。
- 现实用途：企业里训练任何模型（哪怕千亿参数），核心动作都是今天这两格——算 loss、backward 出梯度、按梯度改参数。区别只是参数量从 1 个变成几千亿个。
- 学习深度：需要能**手算**一个单参数例子的梯度和一步更新，并解释为什么是「减」不是「加」。不需要现在看懂自动微分的实现。
- 今天的例子是**教学简化**：只有一个参数、没有神经网络。第 6 周会把同一套动作放大到几十万个参数上。

## 格 1：`backward()` 产出了什么

今天只有一个「参数」，叫 `w`，初始值是 `1.0`。

我们希望它变成 `3.0`，所以定义「差距的平方」作为 loss：

```
loss = (w - 3.0) ** 2
```

- `w` 是 `1.0` 时，`loss = (1 - 3)² = 4.0`
- `w` 是 `3.0` 时，`loss = 0`（完美）

### 先预测（运行前想好答案）

1. 跑完 `loss.backward()` 之后，`w.data`（参数本身）**会不会变**？
2. `w.grad` 打印出来，会是**正数还是负数**？大概是多少？
3. 在跑 `backward()` 之前打印 `w.grad`，会看到什么？

把答案写下来再往下跑。

In [5]:
import torch

# 今天唯一的「参数」。requires_grad=True 是在告诉 PyTorch：
# 这个张量需要算梯度，请记录所有和它有关的运算。
w = torch.tensor([1.0], requires_grad=True)

target = 3.0                      # 我们希望 w 变成的值
loss = (w - target) ** 2          # 「差距的平方」

print("=== backward 之前 ===")
print("w.data =", w.data)          # 参数本身的值
print("w.grad =", w.grad)          # 还没算过，这里会是 None
print("loss   =", loss.item())

loss.backward()                   # 这一步只干一件事：算坡度，写进 .grad

print()
print("=== backward 之后 ===")
print("w.data =", w.data, "   <- 参数本身变了吗？")
print("w.grad =", w.grad, "   <- 填进来的是什么？")
print("loss   =", loss.item(), "  <- loss 本身变了吗？")

=== backward 之前 ===
w.data = tensor([1.])
w.grad = None
loss   = 4.0

=== backward 之后 ===
w.data = tensor([1.])    <- 参数本身变了吗？
w.grad = tensor([-4.])    <- 填进来的是什么？
loss   = 4.0   <- loss 本身变了吗？


### 格 1 对照

**手算核对**：`loss = (w - 3)²`，对 `w` 求导得 `2 × (w - 3)`。
代入 `w = 1`：`2 × (1 - 3) = 2 × (-2) = -4`。

打印的 `w.grad = tensor([-4.])` 和手算完全一致。这是今天唯一需要记住的数学。

**三个观察**：

| 打印项 | 结果 | 说明 |
| --- | --- | --- |
| `w.data` | `1.0`，**没变** | `backward()` 不碰参数，只算不改 |
| `w.grad` | `-4.`，从 `None` 变成有值 | 这就是 backward 的产出 |
| `loss` | `4.0`，**没变** | 参数没变，loss 自然也不变 |

**一句话**：`backward()` = 给每个参数填一张「坡度便条」。参数本身原地不动，动的是便条。

**形状对照**：`w` 的形状是 `(1,)`，`w.grad` 的形状也是 `(1,)`——每个参数都有、且只有一份自己的梯度。第 6 周的真实模型有几十万个参数，就有几十万份梯度，形状和参数一一对应。

> 易混淆点：`.data` 是「参数的值」，`.grad` 是「参数的坡度」。它俩是两个不同的张量，`backward()` 只改后者。

## 格 2：坡度便条怎么用

现在手里有：`w = 1.0`，`w.grad = -4.0`。

「梯度下降」的公式只有一行：

```
新的 w = 旧的 w - 学习率 × w.grad
```

取学习率 `lr = 0.1`。

### 先预测

1. `w.grad = -4` 是负数。用一句话说：**w 往哪个方向走，loss 会变大？**
2. 按上面的公式走一步，`w` 会变成多少？loss 会变大还是变小？
3. 如果公式里的减号写成加号（`w + 0.1 × w.grad`），`w` 变成多少？loss 会怎样？

预测完再运行。

In [3]:
lr = 0.1                            # 学习率：这一步迈多大

print("走之前: w =", w.data.item(), "  loss =", loss.item())

# 手动更新参数。.data 表示「直接改数值，不让 PyTorch 记录这次改动」
w.data -= lr * w.grad               # w - 0.1 * (-4) = 1.0 + 0.4 = 1.4

loss2 = (w - target) ** 2           # 用新的 w 重新算一次 loss

print("走一步: w =", w.data.item(), "  loss =", loss2.item())

# 顺手验证「加号会怎样」
w_bad = torch.tensor([1.0], requires_grad=True)
loss_bad = (w_bad - target) ** 2
loss_bad.backward()
w_bad.data += lr * w_bad.grad       # 故意写反
print()
print("如果写成 += : w =", w_bad.data.item(),
      " loss =", ((w_bad - target) ** 2).item(), "  <- 反而变大了")

走之前: w = 1.0   loss = 4.0
走一步: w = 1.399999976158142   loss = 2.56000018119812

如果写成 += : w = 0.6000000238418579  loss = 5.760000228881836   <- 反而变大了


### 格 2 对照

| | w | loss |
| --- | --- | --- |
| 起点 | 1.0 | 4.00 |
| 走一步（`-=`） | **1.4** | **2.56** |
| 写反了（`+=`） | 0.6 | 5.76 |

> 打印出来可能是 `1.399999976158142` 而不是 `1.4`。这不是算错了：PyTorch 默认用 32 位小数存数字，精度有限。看结果保留三四位小数就够，别纠结尾巴。

**负号怎么读**：`w.grad = -4` 说的是「**w 往小走，loss 会变大**」。
反过来就是：loss 想变小，w 得**往大走**（朝 3.0 去）。

**为什么是减**：`w - 0.1 × (-4) = w + 0.4`。减一个负数等于加。公式写「减」，方向由 `.grad` 的正负号自己决定——你不需要判断该往哪边走，减号就够了。

**记忆锚点（数轴）**：

```
  w 的大小轴（右 = w 变大）：

          走之后    走之前              目标
           1.4       1.0                3.0
  ──────────●─────────●──────────────────●──────────→
                      │
                      └─ w.grad = -4 ：往左走，loss 变大
                         ⇒ loss 下降方向 = 往右
                         ⇒ 减掉 -4 = 往右走 0.4

  loss ：  2.56  ←── 变小 ──  4.00
```

**一句话**：`.grad` 指向 loss **上升**的方向；「梯度下降」就是每次都朝它的**反方向**迈一小步。

> 上一课记住的 `gradient = p - y`，今天在这里落了地：把「模型给正确答案的概率」和「正确答案本身」的差距，变成每个参数身上一张有正负号的坡度便条。

---

### 今天到这里

你刚刚手动做完的这两步，就是项目训练循环里的 `trainer/train_pretrain.py` 第 **65** 行：

```python
scaler.scale(loss).backward()     # ← 格 1
```

周围的 `scaler.step(optimizer)`（76 行）和 `optimizer.zero_grad()`（79 行）就是「格 2 的自动版」和「下一次要回答的问题」。`scaler.*` 今天当它透明。

**下次（30 分钟）**：`optimizer.step()` 和 `zero_grad()` —— 为什么梯度必须清零，以及学习率取 0.1 / 1.0 / 3.0 时 loss 曲线长什么样。

## 验收（用自己的话说出来才算过）

1. 跑完 `loss.backward()`，`w.data` 变了吗？变的是哪个东西？
2. `w.grad = -4` 这个负号，用「w 往哪边走 loss 会变大」这句话说一遍。
3. 更新公式为什么写 `-` 而不是 `+`？如果写 `+`，`w` 会走到哪、loss 会怎样？

**答不出来就不进下一课**（日志里的规矩）。把答案说给我听，我记录到 `docs/LEARNING_LOG.md`。